In [1]:
import pandas as pd
import numpy as np

### **UCDP**
---

UCDP Georeferenced Event Dataset (GED) Global version 25.1 spanning from 1989-01 to 2024-12

This dataset is UCDP's most disaggregated dataset, covering individual events of organized violence (phenomena of lethal violence occurring at a given time and place). These events are sufficiently fine-grained to be geo-coded down to the level of individual villages, with temporal durations disaggregated to single, individual days.

- From this file get the <code>country, conflict_id, dyad_id, date_start, date_end, best</code>

Transform the event dataset to a country dataset, expanding the dates.


In [2]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv')
relevant_columns = ['country','conflict_new_id', 'date_start','date_end','best', 'dyad_new_id', 'type_of_violence', 'region', 'conflict_name', 'side_a', 'side_b']
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])

#join with isocode and add isocode
isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep = ';')
df_ucdp = df_ucdp.merge(isocodes[['name', 'alpha_3']], left_on='country', right_on='name', how='left').rename(columns = {'alpha_3':'isocode'})

#filter by type 1 of conflict
df_ucdp = df_ucdp[df_ucdp['type_of_violence']==1].reset_index(drop=True).copy()
df_ucdp

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/164182624.py:1: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv')


,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,region,conflict_name,side_a,side_b,name,isocode
0,Afghanistan,259,2017-07-31 00:00:00.000,2017-07-31 00:00:00.000,6,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
1,Afghanistan,259,2021-08-26 00:00:00.000,2021-08-26 00:00:00.000,183,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
2,Afghanistan,259,2021-08-28 00:00:00.000,2021-08-28 00:00:00.000,2,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
3,Afghanistan,259,2021-08-29 00:00:00.000,2021-08-29 00:00:00.000,10,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
4,Afghanistan,333,1989-01-01 00:00:00.000,1989-01-01 00:00:00.000,4,727,1,Asia,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-30 00:00:00.000,2024-05-30 00:00:00.000,14,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM
271327,Yemen (North Yemen),16099,2024-06-13 00:00:00.000,2024-06-13 00:00:00.000,5,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM
271328,Yemen (North Yemen),16099,2024-09-10 00:00:00.000,2024-09-10 00:00:00.000,2,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM
271329,Yemen (North Yemen),16099,2024-11-12 00:00:00.000,2024-11-12 00:00:00.000,10,17738,1,Middle East,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM


In [3]:
df_ucdp["date_start"] = pd.to_datetime(df_ucdp["date_start"])
df_ucdp["date_end"]   = pd.to_datetime(df_ucdp["date_end"])

df_ucdp["date_start"] = df_ucdp["date_start"].dt.to_period("M").dt.to_timestamp()
df_ucdp["date_end"]   = df_ucdp["date_end"].dt.to_period("M").dt.to_timestamp()


#1. Expand the events that cross multiple months and distribute 'best' value evenly
#------------------------------------------------------------------------------------
expanded_events = []

for _, row in df_ucdp.iterrows():
    start, end = row['date_start'], row['date_end']
    months = pd.date_range(start, end, freq='MS')  # una fila por mes
    n_months = len(months)
    
    tmp = pd.DataFrame({
        'conflict_new_id': row['conflict_new_id'],
        'isocode': row['isocode'],
        'country': row['country'],
        'region': row['region'],
        'dyad_new_id': row['dyad_new_id'],
        'date': months,
        'best': row['best'] / n_months if n_months > 0 else row['best'],
        'conflict_name': row['conflict_name'],
        'side_a': row['side_a'],
        'side_b': row['side_b']
    })
    expanded_events.append(tmp)

df_expanded = pd.concat(expanded_events, ignore_index=True)
df_expanded

,conflict_new_id,isocode,country,region,dyad_new_id,date,best,conflict_name,side_a,side_b
0,259,AFG,Afghanistan,Asia,524,2017-07-01,6.0,Iraq: Government,Government of Iraq,IS
1,259,AFG,Afghanistan,Asia,524,2021-08-01,183.0,Iraq: Government,Government of Iraq,IS
2,259,AFG,Afghanistan,Asia,524,2021-08-01,2.0,Iraq: Government,Government of Iraq,IS
3,259,AFG,Afghanistan,Asia,524,2021-08-01,10.0,Iraq: Government,Government of Iraq,IS
4,333,AFG,Afghanistan,Asia,727,1989-01-01,4.0,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction
...,...,...,...,...,...,...,...,...,...,...
279485,16099,YEM,Yemen (North Yemen),Middle East,17738,2024-05-01,14.0,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen)
279486,16099,YEM,Yemen (North Yemen),Middle East,17738,2024-06-01,5.0,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen)
279487,16099,YEM,Yemen (North Yemen),Middle East,17738,2024-09-01,2.0,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen)
279488,16099,YEM,Yemen (North Yemen),Middle East,17738,2024-11-01,10.0,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen)


In [4]:
#2. Aggregate to country-month level and sum the 'best' values without distinguishing countries
#-----------------------------------------------------------------------------------------------
df_country_month = (
    df_expanded
    .groupby(['isocode','country','date'], as_index=False)
    .agg(
        best=('best','sum'),
        n_conflicts=('conflict_new_id', 'nunique'),
        n_events=('dyad_new_id', 'count'),
        region = ('region','first')    
    )
)
df_country_month = df_country_month.rename(columns={'date':'year_mo'})
df_country_month

,isocode,country,year_mo,best,n_conflicts,n_events,region
0,AFG,Afghanistan,1989-01-01,693.750000,1,13,Asia
1,AFG,Afghanistan,1989-02-01,162.750000,1,18,Asia
2,AFG,Afghanistan,1989-03-01,1745.750000,1,21,Asia
3,AFG,Afghanistan,1989-04-01,495.750000,1,28,Asia
4,AFG,Afghanistan,1989-05-01,441.000000,1,18,Asia
...,...,...,...,...,...,...,...
11597,YEM,Yemen (North Yemen),2024-11-01,26.166667,2,8,Middle East
11598,YEM,Yemen (North Yemen),2024-12-01,61.166667,3,23,Middle East
11599,ZAF,South Africa,1989-02-01,0.000000,1,1,Africa
11600,ZAF,South Africa,1989-04-01,359.000000,1,55,Africa


### **Agreements**



In [5]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv')
df_pax.columns = df_pax.columns.str.lower()
df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

df_pax = (
    df_pax
    .set_index([c for c in df_pax.columns if c not in ['loc1iso', 'loc2iso']])
    [['loc1iso', 'loc2iso']]
    .stack()
    .reset_index()
    .drop(columns='level_278')  # removes column indicating loc1/loc2
    .rename(columns={0: 'isocode'})
)

# Replace SSD -> SDN before July 2011
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)

df_pax["agreement"] = 1
df_pax["subs_agreement"] = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"] = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"] = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"] = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"] = (df_pax["stagesub"].str.lower() == "rel").astype(int)

df_agreements = (
    df_pax
    .groupby(["isocode", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),  # al menos un acuerdo ese mes
        comp_agreement=("comp_agreement", "max"),  # al menos un SubComp
        subs_agreement=("subs_agreement", "max"),
        cea_agreement = ('cea_agreement', "max"), 
        cea_ceamix_agreement = ('cea_ceamix_agreement', "max"),
        cea_ceas_agreement = ('cea_ceas_agreement', "max"),
        cea_rel_agreement = ('cea_rel_agreement', "max"),
        ce=('ce','max')
    )
)
df_agreements

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/1551532066.py:1: DtypeWarning: Columns (23,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv')
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/1551532066.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/1551532066.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.cop

,isocode,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce
0,AFG,1992-04-01,1,0,1,0,0,0,0,0
1,AFG,1993-03-01,1,0,1,0,0,0,0,2
2,AFG,1999-07-01,1,0,0,0,0,0,0,1
3,AFG,2001-12-01,1,0,1,0,0,0,0,0
4,AFG,2002-01-01,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
1408,ZAF,1992-09-01,1,0,0,0,0,0,0,0
1409,ZAF,1992-11-01,1,0,1,0,0,0,0,0
1410,ZAF,1993-11-01,1,1,0,0,0,0,0,0
1411,ZAF,1994-02-01,1,0,1,0,0,0,0,0


In [6]:
df_panel = df_country_month.merge(
    df_agreements,
    on=["isocode", "year_mo"],
    how="left"
)
df_panel[["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']] = df_panel[
    ["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']
].fillna(0).astype(int)

df_panel['isocode_num'] = df_panel['isocode'].astype('category').cat.codes + 1
df_panel['region_num'] = df_panel['region'].astype('category').cat.codes + 1

df_panel

,isocode,country,year_mo,best,n_conflicts,n_events,region,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,isocode_num,region_num
0,AFG,Afghanistan,1989-01-01,693.750000,1,13,Asia,0,0,0,0,0,0,0,0,1,3
1,AFG,Afghanistan,1989-02-01,162.750000,1,18,Asia,0,0,0,0,0,0,0,0,1,3
2,AFG,Afghanistan,1989-03-01,1745.750000,1,21,Asia,0,0,0,0,0,0,0,0,1,3
3,AFG,Afghanistan,1989-04-01,495.750000,1,28,Asia,0,0,0,0,0,0,0,0,1,3
4,AFG,Afghanistan,1989-05-01,441.000000,1,18,Asia,0,0,0,0,0,0,0,0,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11597,YEM,Yemen (North Yemen),2024-11-01,26.166667,2,8,Middle East,0,0,0,0,0,0,0,0,107,5
11598,YEM,Yemen (North Yemen),2024-12-01,61.166667,3,23,Middle East,0,0,0,0,0,0,0,0,107,5
11599,ZAF,South Africa,1989-02-01,0.000000,1,1,Africa,0,0,0,0,0,0,0,0,108,1
11600,ZAF,South Africa,1989-04-01,359.000000,1,55,Africa,0,0,0,0,0,0,0,0,108,1


### Balanced panel generation

This is done after mergin the agreements, so we include only the interventions that occur in violent periods of the conflicts.

In [7]:
#3. Fill missing months with 0 fatalities to expand to a balanced panel
#-----------------------------------------------------------------------------------------------
# Global dataset range (for example)
global_start = df_ucdp.date_start.min()
global_end   = df_ucdp.date_end.max()

iso_to_country = df_panel.drop_duplicates('isocode').set_index('isocode')['country'].to_dict()


# crear grid de país–mes
isocodes = df_panel['isocode'].unique()
all_months = pd.date_range(global_start, global_end, freq='MS')

import itertools
full_index = pd.DataFrame(itertools.product(isocodes, all_months), columns=['isocode', 'year_mo'])

# merge con tus datos y rellenar ceros
df_panel = full_index.merge(df_panel, on=['isocode', 'year_mo'], how='left')
df_panel['country'] = df_panel['isocode'].map(iso_to_country)
df_panel['real_observation'] = np.where(df_panel['best'].notna(), 1, 0)


# Fill in missing values
columns_to_fill = ['best', 'agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement', 'ce']
for i in columns_to_fill:
    df_panel[i] = df_panel.groupby('isocode')[i].fillna(0)

df_panel['log_best'] = np.log1p(df_panel['best'])

columns_to_ffill_bfill = ['isocode', 'country', 'isocode_num', 'region','region_num']
for i in columns_to_ffill_bfill:
    df_panel[i] = df_panel.groupby('isocode')[i].transform(lambda x: x.ffill().bfill())

df_panel


/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/2392698621.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_panel[i] = df_panel.groupby('isocode')[i].fillna(0)
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/2392698621.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_panel[i] = df_panel.groupby('isocode')[i].fillna(0)
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_89726/2392698621.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to

,isocode,year_mo,country,best,n_conflicts,n_events,region,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,isocode_num,region_num,real_observation,log_best
0,AFG,1989-01-01,Afghanistan,693.75,1.0,13.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,6.543552
1,AFG,1989-02-01,Afghanistan,162.75,1.0,18.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,5.098341
2,AFG,1989-03-01,Afghanistan,1745.75,1.0,21.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,7.465512
3,AFG,1989-04-01,Afghanistan,495.75,1.0,28.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,6.208087
4,AFG,1989-05-01,Afghanistan,441.00,1.0,18.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,6.091310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000
46652,ZAF,2024-09-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000
46653,ZAF,2024-10-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000
46654,ZAF,2024-11-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000


### **First treatment date per conflict**
---


In [8]:
df_panel['year_mo'] = pd.to_datetime(df_panel['year_mo'], format='%Y-%m').dt.to_period('M')
base_date = df_panel['year_mo'].min()
#numeric month index starting from 1
df_panel['year_mo_numeric'] = (df_panel["year_mo"] - base_date).apply(lambda x: x.n+1)

for i in ['agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement']:
    #create the group variable for first treatment date
    df_panel[f'first_{i}_date'] = (
    df_panel[df_panel[i] == 1]
    .groupby('isocode')['year_mo_numeric']
    .transform('min')
    )

    df_panel[f'first_{i}_date'] = (
    df_panel.groupby('isocode')[f'first_{i}_date']
    .transform(lambda x: x.ffill().bfill())
    )
    #Variable gvar: 0 = never treated, positive = month of the first treatment
    df_panel[f'first_{i}_date'] = df_panel[f'first_{i}_date'].fillna(0).astype(int)

    #Dummy: indicator for treated units by agreements
    df_panel[f'treated_{i}'] = (
    df_panel.groupby('isocode')[i]
    .transform('max')
    )

df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,agreement,comp_agreement,subs_agreement,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,AFG,1989-01,Afghanistan,693.75,1.0,13.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
1,AFG,1989-02,Afghanistan,162.75,1.0,18.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
2,AFG,1989-03,Afghanistan,1745.75,1.0,21.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
3,AFG,1989-04,Afghanistan,495.75,1.0,28.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
4,AFG,1989-05,Afghanistan,441.00,1.0,18.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46652,ZAF,2024-09,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46653,ZAF,2024-10,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46654,ZAF,2024-11,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [9]:
df_panel.to_csv('../../data/output/country_level/country_panel.csv')